## **Goal**
**Given a image of a human pointing, use MediaPipePose to generate:**
* 2D joint positions in pixel coordinates
* 3D joint positions in camera coordinates (Unreal Engine format)

### **Setup**

```
conda create -n fbv-media-pipe-pose python=3.10
conda activate fbv-media-pipe-pose
pip install mediapipe opencv-python
conda install ipykernel=6.29.5
pip install PyQt5
```

In [61]:
import cv2
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python.vision import drawing_utils
from mediapipe.tasks.python.vision import drawing_styles
from mediapipe.tasks.python import vision

In [62]:
print(f"Using MediaPipePose Version: {mp.__version__}")

Using MediaPipePose Version: 0.10.33


In [63]:
%matplotlib qt

### **Sample Image**

In [64]:
# Load & Preview Sample Image

image = cv2.imread("00001A.png")
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
print(f"Image Shape: {image.shape}")

plt.figure(figsize=(12, 12))
plt.imshow(image)
plt.axis('off')
plt.show()

Image Shape: (1080, 1920, 3)


### **Visualization Helpers**

In [65]:
# Helper Util from: https://ai.google.dev/edge/mediapipe/solutions/vision/pose_landmarker
def draw_landmarks_on_image(rgb_image, detection_result):
    pose_landmarks_list = detection_result.pose_landmarks
    annotated_image = np.copy(rgb_image)

    pose_landmark_style = drawing_styles.get_default_pose_landmarks_style()
    pose_connection_style = drawing_utils.DrawingSpec(color=(0, 255, 0), thickness=2)

    for pose_landmarks in pose_landmarks_list:
        drawing_utils.draw_landmarks(
            image=annotated_image,
            landmark_list=pose_landmarks,
            connections=vision.PoseLandmarksConnections.POSE_LANDMARKS,
            landmark_drawing_spec=pose_landmark_style,
            connection_drawing_spec=pose_connection_style)

    return annotated_image

In [66]:
def plot_world_landmarks(detection_result):
    landmarks = detection_result.pose_world_landmarks[0]
    points = np.array([[lm.x, lm.y, lm.z] for lm in landmarks])

    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(111, projection='3d')

    ax.scatter(points[:, 0], points[:, 2], -points[:, 1], c='r')

    connections = vision.PoseLandmarksConnections.POSE_LANDMARKS
    
    for connection in connections:
        start_idx = connection.start
        end_idx = connection.end
        
        ax.plot(
            [points[start_idx, 0], points[end_idx, 0]],
            [points[start_idx, 2], points[end_idx, 2]],
            [-points[start_idx, 1], -points[end_idx, 1]],
            c='b'
        )
    
    ax.set_aspect('equal')

    ax.set_title('3D World Coordinates (Meters)')
    plt.show()

In [67]:
def draw_reprojected_world_landmarks(rgb_image, world_landmarks_numpy, rvec, tvec, camera_matrix, dist_coeffs):
    annotated_image = np.copy(rgb_image)
    
    points_3d = world_landmarks_numpy[:, :3].astype(np.float32)
    reprojected_pts, _ = cv2.projectPoints(points_3d, rvec, tvec, camera_matrix, dist_coeffs)
    points_2d = reprojected_pts.reshape(-1, 2).astype(int)
    
    connections = vision.PoseLandmarksConnections.POSE_LANDMARKS
    for connection in connections:
        start_idx = connection.start
        end_idx = connection.end
        
        pt1 = tuple(points_2d[start_idx])
        pt2 = tuple(points_2d[end_idx])
        
        cv2.line(annotated_image, pt1, pt2, (0, 255, 0), 2)

    for pt in points_2d:
        cv2.circle(annotated_image, tuple(pt), 4, (0, 0, 255), -1)
        
    return annotated_image

In [68]:
def plot_transformed_world_landmarks(transformed_pts):
    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(111, projection='3d')

    x = transformed_pts[:, 0]
    y = transformed_pts[:, 1]
    z = transformed_pts[:, 2]

    ax.scatter(x, z, -y, c='r', marker='o')

    connections = vision.PoseLandmarksConnections.POSE_LANDMARKS
    
    for connection in connections:
        start_idx = connection.start
        end_idx = connection.end
        
        ax.plot(
            [x[start_idx], x[end_idx]],
            [z[start_idx], z[end_idx]],
            [-y[start_idx], -y[end_idx]],
            c='b'
        )

    ax.set_xlabel('Camera X (Right)')
    ax.set_ylabel('Camera Z (Depth)')
    ax.set_zlabel('Camera Y (Up)')
    
    ax.set_aspect('equal')
    ax.set_title('3D Camera Coordinates (Meters from Lens)')

    plt.show()

### **Pose Regression**

In [69]:
# Create PoseLandmarker Object

base_options = python.BaseOptions(model_asset_path='../models/pose_landmarker_full.task')
options = vision.PoseLandmarkerOptions(base_options=base_options)
detector = vision.PoseLandmarker.create_from_options(options)

In [70]:
# Detect Pose & Show Annotated Image

mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image)
detection_result = detector.detect(mp_image)

In [71]:
# Show Annotated Image (Pixel Coordinates)

annotated_image = draw_landmarks_on_image(mp_image.numpy_view(), detection_result)
plt.figure(figsize=(12, 12))
plt.imshow(annotated_image)
plt.axis('off')
plt.show()

In [72]:
# Show 3D Coordinates

plot_world_landmarks(detection_result)

### **Convert To Numpy**

In [73]:
def landmarks_to_numpy(landmarks):
    if not landmarks:
        return np.zeros((33, 5))

    return np.array([[lm.x, lm.y, lm.z, lm.visibility, lm.presence] for lm in landmarks[0]])
    

In [74]:
# Create Numpy Arrays

pose_landmarks_numpy = landmarks_to_numpy(detection_result.pose_landmarks)
pose_world_landmarks_numpy = landmarks_to_numpy(detection_result.pose_world_landmarks)

In [75]:
# Show Pose Landmarks Output

print(detection_result.pose_landmarks)
print(pose_landmarks_numpy)

[[NormalizedLandmark(x=0.5202733278274536, y=0.471772700548172, z=-0.22479380667209625, visibility=0.9999774694442749, presence=0.9999371767044067, name=None), NormalizedLandmark(x=0.523451566696167, y=0.4635620713233948, z=-0.22247016429901123, visibility=0.9998757839202881, presence=0.9998536109924316, name=None), NormalizedLandmark(x=0.5256140232086182, y=0.463396281003952, z=-0.22243517637252808, visibility=0.999886155128479, presence=0.999854326248169, name=None), NormalizedLandmark(x=0.5276365280151367, y=0.4630993604660034, z=-0.2224241942167282, visibility=0.9998699426651001, presence=0.9998455047607422, name=None), NormalizedLandmark(x=0.5179146528244019, y=0.4621874690055847, z=-0.22059528529644012, visibility=0.9998045563697815, presence=0.9998226761817932, name=None), NormalizedLandmark(x=0.5162586569786072, y=0.46125298738479614, z=-0.22061225771903992, visibility=0.9998036026954651, presence=0.9997825026512146, name=None), NormalizedLandmark(x=0.5150147676467896, y=0.4602

In [76]:
# Show Pose World Landmarks Output

print(detection_result.pose_world_landmarks)
print(pose_world_landmarks_numpy)

[[Landmark(x=0.0025679580867290497, y=-0.5049461126327515, z=-0.37224510312080383, visibility=0.9999774694442749, presence=0.9999371767044067, name=None), Landmark(x=0.01628543995320797, y=-0.5475043654441833, z=-0.3719860315322876, visibility=0.9998757839202881, presence=0.9998536109924316, name=None), Landmark(x=0.016766328364610672, y=-0.5479738712310791, z=-0.3715333342552185, visibility=0.999886155128479, presence=0.999854326248169, name=None), Landmark(x=0.016657844185829163, y=-0.5486213564872742, z=-0.37114420533180237, visibility=0.9998699426651001, presence=0.9998455047607422, name=None), Landmark(x=-0.016225438565015793, y=-0.5476194024085999, z=-0.3665039539337158, visibility=0.9998045563697815, presence=0.9998226761817932, name=None), Landmark(x=-0.01584085449576378, y=-0.5482884049415588, z=-0.3674483895301819, visibility=0.9998036026954651, presence=0.9997825026512146, name=None), Landmark(x=-0.01601386070251465, y=-0.5497327446937561, z=-0.36623191833496094, visibility=

### **Transform Pose World Landmarks To Reproject To Pose Landmarks**

In [77]:
# Get Relevant Coordinates

h, w, _ = image.shape

idx = [0, 11, 12, 14, 16, 23, 24, 25, 26] # Nose, Shoulders, Elbow (Right), Wrist (Right), Hips, Knees

image_pts = pose_landmarks_numpy[idx, :2].astype(np.float32)
image_pts[:, 0] *= w
image_pts[:, 1] *= h

world_pts = pose_world_landmarks_numpy[idx, :3].astype(np.float32)

In [78]:
# Camera Parameters (Intrinsics)

focal_length_mm = 35.0
sensor_width_mm = 36.0 

f_pixel = (focal_length_mm * w) / sensor_width_mm

camera_matrix = np.array([
    [f_pixel, 0, w / 2],
    [0, f_pixel, h / 2],
    [0, 0, 1]
], dtype=np.float32)

dist_coeffs = np.zeros((4, 1))

In [79]:
# Solve Rotation & Translation

success, rvec, tvec = cv2.solvePnP(
    world_pts.reshape(-1, 1, 3), 
    image_pts.reshape(-1, 1, 2), 
    camera_matrix, 
    dist_coeffs, 
    flags=cv2.SOLVEPNP_ITERATIVE
)

In [80]:
# Transform Pose World Landmarks

all_world_pts = pose_world_landmarks_numpy[:, :3].astype(np.float32)
rmat, _ = cv2.Rodrigues(rvec)
transformed_world_pts = (rmat @ all_world_pts.T).T + tvec.T

In [81]:
# Show Annotated Image

reprojected_img = draw_reprojected_world_landmarks(
    image, 
    pose_world_landmarks_numpy, 
    rvec, 
    tvec, 
    camera_matrix, 
    dist_coeffs
)

plt.figure(figsize=(12, 12))
plt.imshow(reprojected_img)
plt.title("Reprojected World Landmarks (Camera Space)")
plt.axis('off')
plt.show()

In [82]:
# Show 3D Coordinates

plot_transformed_world_landmarks(transformed_world_pts)

### **Process Image B Pair**

In [83]:
# Store Image A Pose Coordinates

image_points_2d_A = pose_landmarks_numpy[:, :2]

In [84]:
# Predict Image B Pose Coordinates

image_B = cv2.imread("00001B.png")
image_B = cv2.cvtColor(image_B, cv2.COLOR_BGR2RGB)
mp_image_B = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_B)
detection_result_B = detector.detect(mp_image_B)
pose_landmarks_numpy_B = landmarks_to_numpy(detection_result_B.pose_landmarks)

In [85]:
# Show Annotated Image (B Pose On Image B)

annotated_image = draw_landmarks_on_image(mp_image_B.numpy_view(), detection_result_B)
plt.figure(figsize=(12, 12))
plt.imshow(annotated_image)
plt.axis('off')
plt.show()

In [86]:
# Show Annotated Image (B Pose On Image A)

annotated_image = draw_landmarks_on_image(mp_image.numpy_view(), detection_result_B)
plt.figure(figsize=(12, 12))
plt.imshow(annotated_image)
plt.axis('off')
plt.show()

In [87]:
# Store Image B Pose Coordinates

image_points_2d_B = pose_landmarks_numpy_B[:, :2]

### **Export Coordinates**

In [88]:
# Scale Images Points To Pixels

image_points_2d_A[:, 0] *= w
image_points_2d_A[:, 1] *= h

image_points_2d_B[:, 0] *= w
image_points_2d_B[:, 1] *= h

In [89]:
# Store Pose World Landmards

pose_points_3d = pose_world_landmarks_numpy[:, :3]

In [90]:
# Save Coordinates To Files

np.save('mpp_2d_landmarks_A.npy', image_points_2d_A)
np.save('mpp_2d_landmarks_B.npy', image_points_2d_B)
np.save('mpp_3d_landmarks_A.npy', pose_points_3d)
np.save('mpp_3d_camera_coords_A.npy', transformed_world_pts)

print("Files saved successfully!")
print(f"2D Shape: {image_points_2d_A.shape}") # (33, 2)
print(f"2D Shape: {image_points_2d_B.shape}") # (33, 2)
print(f"3D Shape: {pose_points_3d.shape}") # (33, 3)
print(f"3D Shape: {transformed_world_pts.shape}") # (33, 3)

Files saved successfully!
2D Shape: (33, 2)
2D Shape: (33, 2)
3D Shape: (33, 3)
3D Shape: (33, 3)
